# Oppitunti 08 - Moniagenttinen suunnittelumalli


## Asennus


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

import os
import asyncio
import dotenv

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Miksi monitoimijajärjestelmät?

Todelliset tehtävät, kuten matkan suunnittelu, vaativat monenlaista osaamista — logistiikkaa, paikallistuntemusta, budjetointia ja muuta. Yksi agentti, joka yrittää hoitaa kaiken, muuttuu nopeasti kömpelöksi.

Monitoimijajärjestelmät ratkaisevat tämän **erikoistumisen** avulla: kukin agentti keskittyy yhteen osa-alueeseen, tuottaen parempilaatuisia tuloksia kuin yleisosaaja. Ne myös parantavat **skaalautuvuutta** — voit lisätä uusia agenteja (esim. lentojen asiantuntija, ravintolakriitikko) ilman, että nykyistä työnkulkua tarvitsee kirjoittaa uudelleen. Agentit muodostavat yhdessä rakenteellisen putkiston, jossa konteksti siirtyy yhdeltä seuraavalle.


## Erikoistuneiden agenttien luominen


In [ ]:
planner_agent = client.as_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = client.as_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## Sekventiaalisen työnkulun rakentaminen

`WorkflowBuilder` antaa sinun yhdistää agenteja suuntautuneeseen graafiin. Tässä luomme yksinkertaisen kahden vaiheen putken: **TravelPlanner** laatii matkasuunnitelman, jonka jälkeen **TravelConcierge** tarkistaa ja parantaa sitä.


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Lisää agenteja työnkulkuun

Yksi moniedustajamallin suurimmista eduista on, kuinka helppoa sitä on laajentaa. Alla lisäämme **BudgetReviewer**-agentin, joka tarkistaa suunnitelman matkustajan budjetin suhteen, merkkaa kohteet, jotka saattavat ylittää kulurajan, ja ehdottaa säästöjä tuovia vaihtoehtoja. Työnkulku suorittaa nyt kolme agenttia peräkkäin:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = client.as_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Yhteenveto

Tässä oppitunnissa opit kuinka:

1. **Luo erikoistuneita agentteja** — jokaisella on oma keskittynyt roolinsa (suunnittelu, concierge, budjetin tarkastus).
2. **Kytke agentit peräkkäiseen työnkulkuun** käyttäen `WorkflowBuilder`-luokkaa ja `add_edge`-metodia.
3. **Suoratoista tuloste** monen agentin putkistosta ja seuraa, mikä agentti puhuu.
4. **Laajenna työnkulkua** lisäämällä uusia agentteja ketjuun muuttamatta olemassa olevia.

Moniagenttinen suunnittelumalli pitää jokaisen agentin yksinkertaisena, mutta tuottaa silti rikkaampia ja perusteellisemmin tarkistettuja tuloksia kuin yksittäinen agentti voi yksin saavuttaa.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vastuuvapauslauseke**:
Tämä asiakirja on käännetty käyttämällä tekoälypohjaista käännöspalvelua [Co-op Translator](https://github.com/Azure/co-op-translator). Vaikka pyrimme tarkkuuteen, otathan huomioon, että automaattiset käännökset saattavat sisältää virheitä tai epätarkkuuksia. Alkuperäinen asiakirja sen alkuperäiskielellä on virallinen lähde. Tärkeissä asioissa suositellaan ammattimaista ihmiskäännöstä. Emme ole vastuussa tämän käännöksen käytöstä aiheutuvista väärinymmärryksistä tai tulkinnoista.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
